# M2 Notebook 03 — Logistic Regression

**Status:** Runnable first edition

## Learning objectives

- Model binary outcomes probabilistically.
- Understand sigmoid and log loss.
- Evaluate thresholds and confusion matrices.

In [ ]:
from srai_math.utils import environment_info,set_seed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
set_seed(42)
environment_info()
from srai_ml import (
    LogisticRegressionGD,accuracy_score,confusion_matrix,f1_score,
    log_loss,precision_score,recall_score,train_test_split,
)


## Logistic model

\[
P(Y=1\mid x)=\sigma(\beta_0+x^\top\beta),
\qquad
\sigma(z)=\frac{1}{1+e^{-z}}.
\]


In [ ]:
rng=np.random.default_rng(7)
X=rng.normal(size=(500,2))
logit=-.3+1.8*X[:,0]-1.2*X[:,1]
p=1/(1+np.exp(-logit))
y=rng.binomial(1,p)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,seed=7)
m=LogisticRegressionGD(learning_rate=.1,max_iter=3000,l2=.001).fit(Xtr,ytr)
prob=m.predict_proba(Xte)[:,1]
pred=m.predict(Xte)
{"log_loss":log_loss(yte,prob),"accuracy":accuracy_score(yte,pred)}


In [ ]:
pd.DataFrame(confusion_matrix(yte,pred),
             index=["Actual 0","Actual 1"],columns=["Predicted 0","Predicted 1"])


## Threshold trade-offs

In [ ]:
rows=[]
for threshold in [.3,.5,.7]:
    pr=(prob>=threshold).astype(int)
    rows.append({"threshold":threshold,"precision":precision_score(yte,pr),
                 "recall":recall_score(yte,pr),"F1":f1_score(yte,pr)})
pd.DataFrame(rows)


In [ ]:
fig,ax=plt.subplots(figsize=(7,4))
ax.plot(m.history_)
ax.set_xlabel("Iteration"); ax.set_ylabel("Log loss")
ax.set_title("Logistic Regression Training")
plt.show()


## Decision Intelligence case

Threshold selection should reflect false-positive and false-negative costs, not habit.

## Key insight

Logistic regression separates probability estimation from the downstream decision threshold.